# ORCA - QLoRA train on **Kaggle GPU** (400 samples ZIP)

## Setup (one time)
1. Kaggle.com -> **Create** -> **New notebook**
2. **Settings** -> Accelerator = **GPU**
3. **Settings** -> Internet = **On**
4. **Add Input** -> dataset with `train.jsonl` + `val.jsonl`
5. Run all cells

## OOM-safe defaults
max_seq_length=1024, LoRA r=16, gradient_checkpointing=True, eval off


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    torch.cuda.empty_cache()
else:
    raise SystemExit('Enable GPU: Settings -> Accelerator -> GPU')


In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf


## 1. Locate train.jsonl / val.jsonl under `/kaggle/input`


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists(), 'No /kaggle/input - use Add Input to attach your dataset'

train_path = None
val_path = None
found = list(INPUT_ROOT.rglob('*.jsonl'))
print('jsonl files found:')
for p in found:
    print(' ', p)

for p in found:
    name = p.name.lower()
    if name == 'train.jsonl':
        train_path = p
    elif name == 'val.jsonl':
        val_path = p

if train_path is None:
    for p in found:
        if 'train' in p.name.lower():
            train_path = p
            break
if train_path is None and found:
    train_path = found[0]
    print('WARN: using first jsonl as train:', train_path)

assert train_path is not None, 'train.jsonl not found. Add Input with train.jsonl'
print('train:', train_path)
print('val:', val_path if val_path else '(none)')

n_train = sum(1 for line in open(train_path, encoding='utf-8') if line.strip())
n_val = sum(1 for line in open(val_path, encoding='utf-8') if line.strip()) if val_path else 0
print('N train:', n_train, '| N val:', n_val)


## 2. Load dataset


In [ ]:
from datasets import load_dataset

data_files = {'train': str(train_path)}
if val_path is not None:
    data_files['validation'] = str(val_path)

raw = load_dataset('json', data_files=data_files)
print(raw)
sample = raw['train'][0]
print('roles:', [m['role'] for m in sample['messages']])
print('preview:', sample['messages'][-1]['content'][:200])


## 3. Qwen2.5-7B 4-bit + LoRA r=16


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
torch.cuda.empty_cache()


## 4. Train 3 epochs (OOM-safe)


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )

args = SFTConfig(
    output_dir='/kaggle/working/orca-400-out',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1.5e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='no',
    bf16=True,
    optim='paged_adamw_8bit',
    max_seq_length=1024,
    packing=False,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_pin_memory=False,
)

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=raw['train'],
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
trainer.train()
print('TRAIN DONE')


## 5. Save adapter + zip under `/kaggle/working`

Download from Output panel after run.


In [ ]:
from pathlib import Path
import shutil

OUT = Path('/kaggle/working/orca-analyst-lora')
if OUT.exists():
    shutil.rmtree(OUT)
model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))

!cd /kaggle/working && zip -r orca-analyst-lora.zip orca-analyst-lora
!ls -lh /kaggle/working/orca-analyst-lora.zip
print('Download: /kaggle/working/orca-analyst-lora.zip')


## 6. Next: vLLM on GPU server

```bash
unzip orca-analyst-lora.zip
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-lora \
  --lora-modules orca-analyst-v1=./orca-analyst-lora \
  --host 0.0.0.0 --port 8000 --max-model-len 4096
```

```bash
AI_BASE_URL=https://YOUR_HOST/v1
AI_MODEL_ANALYSIS=orca-analyst-v1
```
